# Step 4: Attention

<!-- notebook-only -->

> **How to use this notebook**
> - **Just reading?** The text cells contain the whole lesson. You can skip every code cell. An even nicer way to read: the website, built with `.venv/bin/python website/build.py --serve`.
> - **Want to practice?** Run the code cells from top to bottom (Shift+Enter). They recompute every table and let you experiment. In **Your turn** cells, replace `None` with your answer: `check(...)` says if it is right and gives a hint if not, and `solution("4.1")` shows the worked answer.
> - Every question also has a **Show the answer** box you can open.

<!-- /notebook-only -->

**Goal:** let each token look back at the earlier tokens and take what it needs from them, so that `_want` can find `_hungry`.

In [ ]:
# Run this first (practice only): imports, and our sentence as 8 input vectors (Steps 1 to 3)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "toddler").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from toddler.bpe import BPETokenizer
from toddler.data import CORPUS
from toddler.embeddings import COLUMNS, toddler_table
from toddler.exercises import check, solution
from toddler.positions import POSITION_COLUMNS, with_positions


def short(x):
    return "0" if abs(x) < 1e-9 else f"{x:.4f}".rstrip("0").rstrip(".")


np.set_printoptions(formatter={"float_kind": short})
pd.set_option("display.float_format", short)

STREAM = COLUMNS + POSITION_COLUMNS          # the 10 numbers of every vector
tok = BPETokenizer.train(CORPUS)
table = toddler_table(tok)
ids = tok.encode("I am hungry . I want milk .")
inputs = with_positions(table[ids])          # X: 8 rows of 10 numbers, as in Step 3
labels = [f"#{p} {tok.vocab[i]}" for p, i in enumerate(ids, start=1)]
pd.DataFrame(inputs, index=labels, columns=STREAM)

## 4.1 The problem: `_want` doesn't know the child is hungry

After Step 3, our sentence is a table of numbers with **8 rows** (one per token) and **10 columns** (8 for meaning, 2 for position). In math, a table of numbers is called a **matrix**, and its size is written rows × columns: this one is an **8 × 10** matrix (say "8 by 10"). We call it **X**:

```
              me   be want stop feel food  hug sleep    px    py
#1 _I          1    0    0    0    0    0    0     0     1     0
#2 _am         0    1    0    0    0    0    0     0  0.71  0.71
#3 _hungry     0    0    0    0    1    1    0     0     0     1
#4 _.          0    0    0    1    0    0    0     0 -0.71  0.71
#5 _I          1    0    0    0    0    0    0     0    -1     0
#6 _want       0    0    1    0    0    0    0     0 -0.71 -0.71
#7 _milk       0    0    0    0    0    1    0     0     0    -1
#8 _.          0    0    0    1    0    0    0     0  0.71 -0.71
```

Each row is the vector of one token. Look at row 6, `_want`: a 1 in `want`, its position, and nothing else. Nothing in it says "hungry". But to guess `_milk`, the toddler must know the feeling, and the feeling is in row 3, three tokens back.

Until now, every step worked on each row alone. **Attention** is the first step where the rows talk to each other: each token looks back at the earlier tokens and takes what it needs from them.

## 4.2 The idea: a question, a label and a content

Think of a classroom. `_want` stands up and asks: **"Who is a feeling?"** Every token before it holds up a label, and `_hungry`'s label says **"I'm a feeling!"**. So `_want` listens mostly to `_hungry`, which gives it its content: **"food"**.

Every token makes three small vectors from its own 10 numbers:

| the vector | its job | in our story |
|---|---|---|
| **query** | what am I looking for? | "Who is a feeling?" |
| **key** | what do I have? (my label) | "I'm a feeling!" |
| **value** | what do I give to whoever listens to me? | "food" |

**Why not just use the dot product of Step 2?** It finds the tokens that **look like** `_want`, and `_hungry` doesn't: `_want · _hungry` = 0 for the 8 meaning columns, plus (−0.71) × 0 + (−0.71) × 1 for the 2 position columns: **−0.71**. The best match of `_want` would be itself (`_want · _want` = 2). But `_want` needs something **different** from itself: a feeling. That's why each token makes a separate question (query) and label (key): the question can ask for something the token doesn't have. And what a token gives (its value, "food") can differ from what its label says ("I'm a feeling!").

Then each token makes four moves:
1. **Score:** compare its query with the key of every token it can see (dot products, Step 2).
2. **Weights:** turn the scores into shares that add up to 100%.
3. **Mix:** add up the values, each one multiplied by its share.
4. **Add** the mix to its own vector.

All 8 tokens make these moves at the same time, so we compute with whole matrices, one row per token. Here is the full plan, with the size of every matrix (rows × columns):

| move | matrix | size | one row of it is | section |
|---|---|---|---|---|
| start | X | 8 × 10 | the 10 numbers of a token | 4.1 |
| make the 3 vectors | Q = X · Wq, K = X · Wk, V = X · Wv | 8 × 4 each | the query, key and value of a token | 4.3 |
| 1. score | Q · Kᵀ ÷ √4, future hidden | 8 × 8 | the scores of a token for every token | 4.4 |
| 2. weights | softmax of each row | 8 × 8 | the shares of a token (100% in total) | 4.5 |
| 3. mix | weights · V | 8 × 4 | the mix of values that a token gets | 4.6, 4.7 |
| 4. add | X + mix · Wo | 8 × 10 | the new 10 numbers of a token | 4.6, 4.7 |

**Wq**, **Wk**, **Wv** (10 × 4 each) and **Wo** (4 × 10) are the **weight matrices** of attention: the numbers that training learns (Step 7). Everything else is computed from X and these four.

## 4.3 Query, key and value: three matrix multiplications

**The three weight matrices.** Each one has **10 rows**, one for each column of X (`me` … `py`), and **4 columns**, one for each number of the vector it makes. We call these 4 columns **slots**: s1 to s4. In a real model, training sets all these numbers (Step 7). This step uses **our hint**: matrices set by hand, almost all zeros. Here they are in full, side by side:

```
         Wq (query)         Wk (key)           Wv (value)
         s1  s2  s3  s4     s1  s2  s3  s4     s1  s2  s3  s4
me        0   0   0   0      0   0   0   0      0   0   0   0
be        0   0   0   0      0   0   0   0      0   0   0   0
want      6   0   0   0      0   0   0   0      0   0   0   0
stop      0   0   0   0      0   0   0   0      0   0   0   0
feel      0   0   0   0      2   0   0   0      0   0   0   0
food      0   0   0   0      0   0   0   0      1   0   0   0
hug       0   0   0   0      0   0   0   0      0   1   0   0
sleep     0   0   0   0      0   0   0   0      0   0   1   0
px        0   0   0   0      0   0   0   0      0   0   0   0
py        0   0   0   0      0   0   0   0      0   0   0   0
```

How to read them:
- **Wq** has a single non-zero number: **6**, in row `want`, slot s1. So only a token with something in its `want` column asks a question. The 6 sets how **loud** the question is (exercise 4.1 makes it quieter).
- **Wk** has a single non-zero number too: **2**, in row `feel`, slot s1. So only feelings show a label, and it is in **the same slot** as the question. That's what will make them meet in 4.4.
- **Wv** copies the topic of a token into a slot: `food` → s1, `hug` → s2, `sleep` → s3. Slot s4 stays empty: our hint doesn't need it.
- Rows `px` and `py` are all 0: this head ignores positions (section 4.8 shows a head that uses them).

In [ ]:
# Try it: the three weight matrices of our hint (10 rows × 4 slots each), side by side
HEAD_SIZE = 4
SLOTS = [f"s{slot}" for slot in range(1, HEAD_SIZE + 1)]
Wq, Wk, Wv = (np.zeros((len(STREAM), HEAD_SIZE)) for _ in range(3))
Wq[STREAM.index("want"), 0] = 6
Wk[STREAM.index("feel"), 0] = 2
for slot, topic in enumerate(["food", "hug", "sleep"]):
    Wv[STREAM.index(topic), slot] = 1

pd.concat({"Wq": pd.DataFrame(Wq, index=STREAM, columns=SLOTS),
           "Wk": pd.DataFrame(Wk, index=STREAM, columns=SLOTS),
           "Wv": pd.DataFrame(Wv, index=STREAM, columns=SLOTS)}, axis=1)

**How to multiply two matrices.** There is a single rule: **each number of the result is the dot product (Step 2) of one row of the left matrix with one column of the right matrix.**

So slot s1 of the query of `_want` is: row 6 of X · column s1 of Wq. Multiply them number by number, then add up:

```
                    me   be want stop feel food  hug sleep    px    py
row 6 of X           0    0    1    0    0    0    0     0 -0.71 -0.71
column s1 of Wq      0    0    6    0    0    0    0     0     0     0
multiply             0    0    6    0    0    0    0     0     0     0   → sum = 6
```

Slots s2, s3 and s4 use the columns s2, s3 and s4 of Wq. These are all zeros, so every product is 0, and so is every sum. The query of `_want` is **(6, 0, 0, 0)**.

Do the same with each of the 8 rows of X, and you get 8 queries: the matrix **Q**, with one row per token and one column per slot. We write it **Q = X · Wq**, and the sizes show how it works:

```
X (8 × 10)  ·  Wq (10 × 4)  =  Q (8 × 4)
```

- The two **inner** sizes must be equal (10 and 10): a row of X and a column of Wq need the same length for their dot product.
- The result gets the two **outer** sizes: 8 rows (one per token) and 4 columns (one per slot).

![X · Wq = Q: row 6 of X and column s1 of Wq make the framed number of Q](figures/04_x_times_wq.png)

**A shortcut for matrices full of zeros.** A row of the result is also the sum of the rows of Wq, each one multiplied by the matching number of the token. For `_want`: 1 × (row `want`) + (−0.71) × (row `px`) + (−0.71) × (row `py`) = 1 × (6, 0, 0, 0) + (−0.71) × (0, 0, 0, 0) + (−0.71) × (0, 0, 0, 0) = (6, 0, 0, 0). Same answer. `want` is the only non-zero row of Wq, so every token with a 0 in `want` (all of them except `_want`) gets the query (0, 0, 0, 0).

**The key and the value of `_hungry`**, with the same rule. Its key, slot s1: row 3 of X · column s1 of Wk:

```
                    me   be want stop feel food  hug sleep    px    py
row 3 of X           0    0    0    0    1    1    0     0     0     1
column s1 of Wk      0    0    0    0    2    0    0     0     0     0
multiply             0    0    0    0    2    0    0     0     0     0   → sum = 2
```

Its value, slot s1: row 3 of X · column s1 of Wv:

```
                    me   be want stop feel food  hug sleep    px    py
row 3 of X           0    0    0    0    1    1    0     0     0     1
column s1 of Wv      0    0    0    0    0    1    0     0     0     0
multiply             0    0    0    0    0    1    0     0     0     0   → sum = 1
```

All its other slots give 0. So `_hungry` has the key **(2, 0, 0, 0)**, "I'm a feeling!", and the value **(1, 0, 0, 0)**, "food".

**The results: three 8 × 4 matrices**, one row per token:

```
             Q = X · Wq         K = X · Wk         V = X · Wv
             s1  s2  s3  s4     s1  s2  s3  s4     s1  s2  s3  s4
#1 _I         0   0   0   0      0   0   0   0      0   0   0   0
#2 _am        0   0   0   0      0   0   0   0      0   0   0   0
#3 _hungry    0   0   0   0      2   0   0   0      1   0   0   0
#4 _.         0   0   0   0      0   0   0   0      0   0   0   0
#5 _I         0   0   0   0      0   0   0   0      0   0   0   0
#6 _want      6   0   0   0      0   0   0   0      0   0   0   0
#7 _milk      0   0   0   0      0   0   0   0      1   0   0   0
#8 _.         0   0   0   0      0   0   0   0      0   0   0   0
```

- **Q:** only `_want` asks a question. It is the only token with a 1 in `want`.
- **K:** only `_hungry` shows a label. It is the only feeling.
- **V:** `_hungry` and `_milk` both offer "food", because both have a 1 in `food`. But `_want` will never take the food of `_milk`: `_milk` shows no label (its key is 0), and it comes after `_want` (section 4.4).

In [ ]:
# Try it: the query of _want, slot by slot (row 6 of X · each column of Wq), then Q, K and V in full
for slot in range(HEAD_SIZE):
    products = inputs[5] * Wq[:, slot]  # row 6 is index 5: Python counts from 0
    print(f"slot s{slot + 1}: {products} → sum = {short(products.sum())}")

queries, keys, values = inputs @ Wq, inputs @ Wk, inputs @ Wv  # Q, K and V (inputs is X; @ multiplies matrices)
pd.concat({"Q = X · Wq": pd.DataFrame(queries, index=labels, columns=SLOTS),
           "K = X · Wk": pd.DataFrame(keys, index=labels, columns=SLOTS),
           "V = X · Wv": pd.DataFrame(values, index=labels, columns=SLOTS)}, axis=1)

## 4.4 Scores: every question meets every label

The score of a token for another token is its query · the key of the other token, divided by √4:

**score = (query · key) ÷ √4**

We need a score for every pair: 8 tokens asking × 8 tokens to look at = 64 scores, so an **8 × 8** matrix. One matrix multiplication computes them all. Step by step:

**Step 1: turn K on its side.** The rule pairs a row of the left matrix with a **column** of the right matrix. The queries are the rows of Q: good. But the keys are the rows of K, and we need them as columns. So we flip K: its row 1 becomes column 1, its row 2 becomes column 2, and so on. This is called the **transpose** of K, written **Kᵀ**. It is a 4 × 8 matrix, and its column 3 is the key of #3 `_hungry`:

```
      #1  #2  #3  #4  #5  #6  #7  #8
s1     0   0   2   0   0   0   0   0
s2     0   0   0   0   0   0   0   0
s3     0   0   0   0   0   0   0   0
s4     0   0   0   0   0   0   0   0
```

**Step 2: multiply Q by Kᵀ.** The sizes fit: Q (8 × 4) · Kᵀ (4 × 8) = 8 × 8. The inner sizes are equal (a query and a key both have 4 numbers), and the result has **one row per token asking** and **one column per token looked at**.

The number in row 6, column 3 is row 6 of Q · column 3 of Kᵀ, that is, the query of `_want` · the key of `_hungry`:

```
                      s1   s2   s3   s4
query of #6 _want      6    0    0    0
key of #3 _hungry      2    0    0    0
multiply              12    0    0    0   → sum = 12
```

Every other pair gives 0: 7 of the 8 queries are (0, 0, 0, 0), and so are 7 of the 8 keys. Here is the whole **Q · Kᵀ**:

```
             #1  #2  #3  #4  #5  #6  #7  #8   ← the token looked at (its key)
#1 _I         0   0   0   0   0   0   0   0
#2 _am        0   0   0   0   0   0   0   0
#3 _hungry    0   0   0   0   0   0   0   0
#4 _.         0   0   0   0   0   0   0   0
#5 _I         0   0   0   0   0   0   0   0
#6 _want      0   0  12   0   0   0   0   0   ← _want meets _hungry
#7 _milk      0   0   0   0   0   0   0   0
#8 _.         0   0   0   0   0   0   0   0
↑ the token asking (its query)
```

![Q · Kᵀ: row 6 of Q and column 3 of Kᵀ make the framed 12](figures/04_q_times_kt.png)

In [ ]:
# Try it: Kᵀ (K turned on its side), the dot product of row 6 and column 3, then the whole Q · Kᵀ
numbers = [f"#{p}" for p in range(1, 9)]
print("Kᵀ (4 × 8):")
print(pd.DataFrame(keys.T, index=SLOTS, columns=numbers))
print("\nrow 6 of Q · column 3 of Kᵀ =", queries[5], "·", keys.T[:, 2], "=", short(queries[5] @ keys.T[:, 2]))
raw_scores = queries @ keys.T  # 8 × 8: one row per token asking, one column per token looked at
pd.DataFrame(raw_scores, index=labels, columns=numbers)

**Step 3: divide by √4.** √4 = 2, so every score is halved: the 12 becomes **6**, and the zeros stay 0. This keeps the scores from growing too big when the vectors get long (exercise 4.6 asks why that matters). Real models also divide by the square root of their key size: √64 = 8, √128 ≈ 11.3.

**Step 4: hide the future (the causal mask).** A token can only look at itself and at the tokens **before** it. When the toddler guesses what comes after `_want`, `_milk` hasn't been said yet, so it is hidden. This is called the **causal mask**: no peeking at the future. Row 1 sees column 1 only, row 2 sees columns 1 and 2, …, and row 8 sees all 8 columns: a staircase. Everything to the right of the diagonal is the future:

| can see? | #1 | #2 | #3 | #4 | #5 | #6 | #7 | #8 |
|---|---|---|---|---|---|---|---|---|
| #1 `_I` | ✓ | ✗ | ✗ | ✗ | ✗ | ✗ | ✗ | ✗ |
| #2 `_am` | ✓ | ✓ | ✗ | ✗ | ✗ | ✗ | ✗ | ✗ |
| #3 `_hungry` | ✓ | ✓ | ✓ | ✗ | ✗ | ✗ | ✗ | ✗ |
| #4 `_.` | ✓ | ✓ | ✓ | ✓ | ✗ | ✗ | ✗ | ✗ |
| #5 `_I` | ✓ | ✓ | ✓ | ✓ | ✓ | ✗ | ✗ | ✗ |
| #6 `_want` | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✗ | ✗ |
| #7 `_milk` | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✗ |
| #8 `_.` | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |

Every ✗ gets the score **−∞** (minus infinity; Python prints it `-inf`). In 4.5, softmax turns the scores into shares, and −∞ always gives a share of exactly 0: a hidden token gets no attention at all. Why not just leave the hidden cells out? Then every row would have a different length. With −∞, all 8 rows are computed the same way, at once.

**The scores, ready for softmax** (8 × 8):

```
               #1    #2    #3    #4    #5    #6    #7    #8
#1 _I           0  -inf  -inf  -inf  -inf  -inf  -inf  -inf
#2 _am          0     0  -inf  -inf  -inf  -inf  -inf  -inf
#3 _hungry      0     0     0  -inf  -inf  -inf  -inf  -inf
#4 _.           0     0     0     0  -inf  -inf  -inf  -inf
#5 _I           0     0     0     0     0  -inf  -inf  -inf
#6 _want        0     0     6     0     0     0  -inf  -inf
#7 _milk        0     0     0     0     0     0     0  -inf
#8 _.           0     0     0     0     0     0     0     0
```

Each row is one token looking back. Row 6 is `_want`, the one we follow:

| token it looks at | #1 `_I` | #2 `_am` | #3 `_hungry` | #4 `_.` | #5 `_I` | #6 `_want` | #7 `_milk` | #8 `_.` |
|---|---|---|---|---|---|---|---|---|
| query · key | 6 × 0 = 0 | 0 | 6 × 2 = 12 | 0 | 0 | 0 | hidden | hidden |
| ÷ 2 = score | 0 | 0 | **6** | 0 | 0 | 0 | −∞ | −∞ |

In all the other rows, the visible scores are 0: these tokens ask no question (section 4.7 shows what they do then).

In [ ]:
# Try it: divide by √4, then hide the future with -inf
scores = raw_scores / np.sqrt(HEAD_SIZE)
future = np.triu(np.ones(scores.shape, dtype=bool), k=1)  # True right of the diagonal: the later tokens
print(pd.DataFrame(np.where(future, "✗", "✓"), index=labels, columns=numbers))
masked_scores = np.where(future, -np.inf, scores)
pd.DataFrame(masked_scores, index=labels, columns=numbers)

## 4.5 From scores to shares: softmax

The scores must become shares that add up to 100%. The recipe is called **softmax**, and we will meet it again in Step 6. It works on **each row separately**: every token shares out its own attention.

1. Raise e (2.718…) to the power of each score. The result is always positive, and a bigger score gives a much bigger result. For a hidden token, e^−∞ = **0**.
2. Divide each result by the total of the row.

Row 6 of the scores, `_want`:

| | #1 `_I` | #2 `_am` | #3 `_hungry` | #4 `_.` | #5 `_I` | #6 `_want` | #7 `_milk` | #8 `_.` | total |
|---|---|---|---|---|---|---|---|---|---|
| score | 0 | 0 | 6 | 0 | 0 | 0 | −∞ | −∞ | |
| e^score | 1 | 1 | 403.4 | 1 | 1 | 1 | 0 | 0 | 408.4 |
| ÷ 408.4 = weight | 0.24% | 0.24% | **98.8%** | 0.24% | 0.24% | 0.24% | 0% | 0% | 100% |

`_want` gives **98.8%** of its attention to `_hungry`, and exactly 0% to `_milk` and `_.`, which are hidden.

Softmax does the same with each of the 8 rows. The result is the **weights** matrix: 8 × 8 like the scores, and each of its rows adds up to 100% (section 4.7 shows it whole). In the matrices below, the weights are written as decimals: 0.9878 is 98.78%, and 0.0024 is 0.24%.

![_want looks back: the key, value, score and weight of every token](figures/04_want_looks_back.png)

In [ ]:
# Try it: softmax on row 6 of the scores (_want), with the future hidden
def softmax(scores):
    exps = np.exp(scores - scores.max())  # subtracting the max avoids huge numbers and changes nothing
    return exps / exps.sum()


want = 5                                  # row 6 is index 5: Python counts from 0
want_scores = masked_scores[want]         # -inf for #7 and #8: the future
want_weights = softmax(want_scores)
print("total of the e^score:", short(np.exp(want_scores).sum()))
pd.DataFrame({"score": want_scores, "e^score": np.exp(want_scores), "weight": want_weights}, index=labels)

## 4.6 Mixing the values, and adding the result

**The mix.** Each value is multiplied by its weight, and the results are added up. With matrices, that's one more multiplication: row 6 of the weights (1 × 8: the 8 weights of `_want`) · V (8 × 4) = the **mix** of `_want` (1 × 4). Its slot s1 is row 6 of the weights · column s1 of V:

```
                      #1     #2     #3     #4     #5     #6     #7     #8
row 6 of weights  0.0024 0.0024 0.9878 0.0024 0.0024 0.0024      0      0
column s1 of V         0      0      1      0      0      0      1      0
multiply               0      0 0.9878      0      0      0      0      0   → sum = 0.9878
```

`_milk` (#7) offers food too (a 1 in column s1 of V), but its weight is 0: the causal mask at work. Columns s2, s3 and s4 of V are all zeros, so the mix of `_want` is **(0.9878, 0, 0, 0)**: almost exactly the value of `_hungry`, (1, 0, 0, 0).

**Back to 10 columns.** The mix has 4 numbers, but the vector of `_want` has 10. The fourth weight matrix, **Wo** (the output matrix, 4 × 10), puts each slot back into a column of X. In our hint, s1 goes to `food`, s2 to `hug` and s3 to `sleep`:

```
         me   be want stop feel food  hug sleep    px    py
s1        0    0    0    0    0    1    0     0     0     0
s2        0    0    0    0    0    0    1     0     0     0
s3        0    0    0    0    0    0    0     1     0     0
s4        0    0    0    0    0    0    0     0     0     0
```

The mix (1 × 4) · Wo (4 × 10) = 10 numbers. With the shortcut of 4.3, that's 0.9878 × (row s1 of Wo) + 0 × (row s2) + 0 × (row s3) + 0 × (row s4): 0.9878 in the `food` column, and 0 everywhere else.

**The add.** These 10 numbers are **added** to the vector of `_want`:

```
                me   be want stop feel  food  hug sleep    px    py
_want            0    0    1    0    0     0    0     0 -0.71 -0.71
+ attention      0    0    0    0    0  0.99    0     0     0     0
= new _want      0    0    1    0    0  0.99    0     0 -0.71 -0.71
```

`_want` now carries "food": the child wants something about food. In sentence 2, `_want` finds `_sad` and takes `hug`; in sentence 3, it finds `_tired` and takes `sleep`.

In [ ]:
# Try it: the mix of _want (row 6 of the weights · V), back to 10 columns (· Wo), then added to _want
Wo = np.zeros((HEAD_SIZE, len(STREAM)))
for slot, topic in enumerate(["food", "hug", "sleep"]):
    Wo[slot, STREAM.index(topic)] = 1

mix = want_weights @ values               # (1 × 8) · (8 × 4): 4 numbers
added = mix @ Wo                          # (1 × 4) · (4 × 10): 10 numbers
print("mix of _want:", mix)
print("\nWo (4 × 10):")
print(pd.DataFrame(Wo, index=SLOTS, columns=STREAM))
pd.DataFrame([inputs[want], added, inputs[want] + added], index=["_want", "+ attention", "= new _want"],
             columns=STREAM)

## 4.7 Every token at once

All 8 tokens make the same moves at the same time, so the whole sentence goes through the same few matrix multiplications. The scores are already done for every token (4.4). Here is the rest, for all 8 rows at once.

**The weights: softmax of each row of the scores** (8 × 8). Each row adds up to 1, that is 100%:

```
                 #1      #2      #3      #4      #5      #6      #7      #8
#1 _I             1       0       0       0       0       0       0       0
#2 _am          0.5     0.5       0       0       0       0       0       0
#3 _hungry   0.3333  0.3333  0.3333       0       0       0       0       0
#4 _.          0.25    0.25    0.25    0.25       0       0       0       0
#5 _I           0.2     0.2     0.2     0.2     0.2       0       0       0
#6 _want     0.0024  0.0024  0.9878  0.0024  0.0024  0.0024       0       0
#7 _milk     0.1429  0.1429  0.1429  0.1429  0.1429  0.1429  0.1429       0
#8 _.         0.125   0.125   0.125   0.125   0.125   0.125   0.125   0.125
```

The same matrix as a picture (darker = more attention):

![The weights: who looks at whom in our sentence](figures/04_attention_pattern.png)

- Row 6: `_want` looks at `_hungry` (0.9878, that is 98.8%).
- Row 1: the first `_I` can only see itself, so it gives itself 100%.
- The other tokens ask no question in our hint: their query is 0, so all their visible scores are 0, and they spread their attention evenly over what they can see: ½ each for `_am`, ⅓ each for `_hungry`, ¼ each for the first `_.`, and so on.

In [ ]:
# Try it: the whole sentence at once, as one function: the matrices of 4.3 to 4.6
def attention(inputs, Wq, Wk, Wv, Wo):
    queries, keys, values = inputs @ Wq, inputs @ Wk, inputs @ Wv        # Q, K, V: 8 × 4 each
    scores = queries @ keys.T / np.sqrt(Wq.shape[1])                     # Q · Kᵀ ÷ √4: 8 × 8
    scores[np.triu(np.ones(scores.shape, dtype=bool), k=1)] = -np.inf    # the future is hidden: e^-inf = 0
    exps = np.exp(scores - scores.max(axis=1, keepdims=True))
    weights = exps / exps.sum(axis=1, keepdims=True)                     # softmax of each row: 8 × 8
    return weights @ values @ Wo, weights                                # mix · Wo (8 × 10), and the weights


added_all, weights = attention(inputs, Wq, Wk, Wv, Wo)
pd.DataFrame(weights, index=labels, columns=numbers)

**The mix of every token: weights · V** (8 × 8 · 8 × 4 = 8 × 4). Row 6 is the mix of `_want` from 4.6, and every other row is made the same way:

```
                 s1  s2  s3  s4
#1 _I             0   0   0   0
#2 _am            0   0   0   0
#3 _hungry   0.3333   0   0   0
#4 _.          0.25   0   0   0
#5 _I           0.2   0   0   0
#6 _want     0.9878   0   0   0
#7 _milk     0.2857   0   0   0
#8 _.          0.25   0   0   0
```

Only slot s1 is non-zero, because only `_hungry` and `_milk` have a value, (1, 0, 0, 0) each. Row 3: `_hungry` gives ⅓ to itself, so it gets ⅓ × food. Row 7: `_milk` gives 1/7 to `_hungry` and 1/7 to itself, two foods: 2/7 = 0.2857.

**What attention adds: mix · Wo** (8 × 4 · 4 × 10 = 8 × 10). Wo puts slot s1 into the `food` column, so only that column changes. Here is the new X, that is X + mix · Wo:

```
              me   be want stop feel  food  hug sleep    px    py
#1 _I          1    0    0    0    0     0    0     0     1     0
#2 _am         0    1    0    0    0     0    0     0  0.71  0.71
#3 _hungry     0    0    0    0    1  1.33    0     0     0     1
#4 _.          0    0    0    1    0  0.25    0     0 -0.71  0.71
#5 _I          1    0    0    0    0   0.2    0     0    -1     0
#6 _want       0    0    1    0    0  0.99    0     0 -0.71 -0.71
#7 _milk       0    0    0    0    0  1.29    0     0     0    -1
#8 _.          0    0    0    1    0  0.25    0     0  0.71 -0.71
```

This is what attention gives back: an 8 × 10 matrix, the same size as X, so that more steps can follow (Step 5). `_want` now carries 0.99 food. The other tokens got a little food too: that's noise from our hand-made hint, and it's harmless for our question, because the guess after "I want" only uses row 6 (Step 6).

In [ ]:
# Try it: the mix of every token (weights · V), then the new X (X + mix · Wo)
print("mix = weights · V (8 × 4):")
print(pd.DataFrame(weights @ values, index=labels, columns=SLOTS))
pd.DataFrame(inputs + added_all, index=labels, columns=STREAM)

In [ ]:
# Try it: in each of the 3 sentences, what attention adds to _want
for sentence in CORPUS:
    sentence_added, _ = attention(with_positions(table[tok.encode(sentence)]), Wq, Wk, Wv, Wo)
    gained = {STREAM[i]: short(v) for i, v in enumerate(sentence_added[5]) if abs(v) > 1e-9}
    print(f"{sentence:<30} → _want gets {gained}")

## 4.8 Several heads, looking for different things

One set of the four weight matrices, Wq, Wk, Wv and Wo, is called a **head**. Real models run many heads side by side, each one looking for something different, and add up what they bring.

For example, a head can look at **the previous token**, using the clock of Step 3:
- its key is the clock hand of the token;
- its query is the clock hand **turned back by 45°** (one step), and made 16 times longer: a sharper question.

Its Wq and Wk only use the rows `px` and `py`. All their other rows are 0:

```
           Wq                     Wk
           s1      s2  s3  s4     s1  s2  s3  s4
px      11.31  -11.31   0   0      1   0   0   0
py      11.31   11.31   0   0      0   1   0   0
```

11.31 is 16 × 0.71: with these numbers, Wq turns the hand back by 45° and makes it 16 times longer. For `_want` (#6), the hand points at 225°: (px, py) = (−0.71, −0.71). With the shortcut of 4.3:

query = (−0.71) × (row `px` of Wq) + (−0.71) × (row `py` of Wq) = **(−16, 0, 0, 0)**: the hand at 180°, 16 times longer.

The hand of #5 `_I` points exactly there: its key is (−1, 0, 0, 0), so its score is 16 ÷ 2 = **8**, the highest. All the scores and weights of `_want` in this head:

| token it looks at | #1 `_I` | #2 `_am` | #3 `_hungry` | #4 `_.` | #5 `_I` | #6 `_want` | #7 `_milk` | #8 `_.` |
|---|---|---|---|---|---|---|---|---|
| its hand points at | 0° | 45° | 90° | 135° | **180°** | 225° | 270° | 315° |
| score | −8 | −5.66 | 0 | 5.66 | **8** | 5.66 | −∞ | −∞ |
| weight | ≈ 0% | ≈ 0% | 0.03% | 8% | **84%** | 8% | 0% | 0% |

Every token from #3 on does the same: 84% to the token just before it. #2 `_am` gives 91% to #1, because it has only 2 tokens to share between, and #1 has no previous token, so it keeps 100% for itself. The cell below computes all the rows.

Our toddler uses a single head, the "feeling finder". Real models have dozens of heads in each block. In papers, the outputs of all the heads are put side by side and multiplied by one big Wo: that's the same as giving each head its own Wo and adding up what they bring.

In [ ]:
# Try it: a second head that looks one step back, with the clock
turn = np.radians(45)
Wq_back, Wk_back = np.zeros((len(STREAM), HEAD_SIZE)), np.zeros((len(STREAM), HEAD_SIZE))
Wq_back[STREAM.index("px"), :2] = 16 * np.array([np.cos(turn), -np.sin(turn)])   # turns the hand back by 45°
Wq_back[STREAM.index("py"), :2] = 16 * np.array([np.sin(turn), np.cos(turn)])
Wk_back[STREAM.index("px"), 0] = 1
Wk_back[STREAM.index("py"), 1] = 1
print(pd.concat({"Wq": pd.DataFrame(Wq_back, index=STREAM, columns=SLOTS),
                 "Wk": pd.DataFrame(Wk_back, index=STREAM, columns=SLOTS)}, axis=1).loc[["px", "py"]])
print("\nquery of _want:", inputs[want] @ Wq_back)
_, weights_back = attention(inputs, Wq_back, Wk_back, Wv, Wo)
pd.DataFrame(weights_back, index=labels, columns=numbers)

## 4.9 How real LLMs do it

The famous formula of the Transformer paper ("Attention Is All You Need", 2017) is exactly what you just computed, for one head:

**attention = softmax(Q · Kᵀ ÷ √(key size) + mask) · V**, with **Q = X · Wq**, **K = X · Wk** and **V = X · Wv**

The mask is 0 where a token can see, and −∞ for the future. Then the result goes through Wo, and is added to X (4.6).

The same matrices, at real size:

| | Toddler | GPT-3 |
|---|---|---|
| X (tokens × numbers per token) | 8 × 10 | up to 2,048 × 12,288 |
| Wq, Wk and Wv of one head | 10 × 4 | 12,288 × 128 |
| scores of one head | 8 × 8 | up to 2,048 × 2,048 |
| heads in each block | 1 | 96 |
| blocks | 1 | 96 |

- **The mask** is the same: when it writes, a model never looks at the tokens that come after.
- **The cost:** the score matrix has one row and one column per token, so the work grows like length × length: twice as many tokens, 4 times as many scores. That's one reason why context windows have a limit.
- **The KV cache:** thanks to the mask, adding a token at the end never changes the rows of the earlier tokens. So while writing, models keep the rows of K and V of the earlier tokens in memory, and only compute one new row of Q, K and V for each new token.
- **Positions:** most recent models turn queries and keys by the clock angle right inside attention (RoPE, Step 3), between X · Wq and Q · Kᵀ, so the scores depend on the distance between tokens.
- What each head looks for is **learned**, never written by hand. Researchers find heads that look at the previous token (like our clock head in 4.8), at the matching bracket, at the name mentioned earlier…

## 4.10 What is still missing

`_want` now carries "food", but nothing has decided "→ milk" yet. Step 5 builds the rest of the transformer block (the feed-forward part, and why everything is *added*). Step 6 turns the final vector into a probability for each token.

<!-- notebook-only -->

The same code is saved in `toddler/attention.py`, and the next notebooks import it from there.

<!-- /notebook-only -->

In [ ]:
# Try it: the package version used by the next notebooks gives the same result
from toddler import attention as toddler_attention

package_added, package_weights = toddler_attention.attention(inputs, **toddler_attention.hint_attention())
assert np.allclose(package_added, added_all) and np.allclose(package_weights, weights)
print("toddler/attention.py gives the same attention")

## Your turn

Do each one **by hand first**, using the matrices above.

### 4.1 If row `want` of Wq were (2, 0, 0, 0) instead of (6, 0, 0, 0), what share of its attention would `_want` give to `_hungry`?

<details>
<summary>Show the answer</summary>
<!-- answer: 0.6 | 0.60 | 0.596 | 0.5964 | 60% | 59.6% -->
<!-- format: a number between 0 and 1, e.g. 0.25 -->

The query of `_want` becomes (2, 0, 0, 0), so the score of `_hungry` becomes 2 × 2 ÷ 2 = **2**. The 5 other visible scores stay 0, and #7 and #8 stay hidden.

e² = 7.39, so the total is 7.39 + 5 × 1 = 12.39, and the share is 7.39 ÷ 12.39 = **0.60**.

A weaker question gives a less focused attention: 60% instead of 98.8%.

</details>

In [ ]:
my_answer = None  # a number between 0 and 1
check("4.1", my_answer)

### 4.2 In "I am sad . I want mommy .", which column of `_want`'s vector does attention add to?

<details>
<summary>Show the answer</summary>
<!-- answer: hug | _hug -->
<!-- format: a column name, e.g. sleep -->

`_want` finds the feeling `_sad` (its key is (2, 0, 0, 0)). The value of `_sad` is (0, 1, 0, 0): slot s2, which Wo puts in the **hug** column. So `_want` gets +0.99 hug.

</details>

In [ ]:
my_answer = None  # a column name, e.g. "sleep"
check("4.2", my_answer)

### 4.3 `_hungry` (#3) has the query (0, 0, 0, 0). What share of its attention does it give to each token it can see?

<details>
<summary>Show the answer</summary>
<!-- answer: 1/3 | 0.33 | 0.333 | 33% | 33.3% -->
<!-- format: a fraction or a number, e.g. 1/4 -->

Row 3 of the scores is 0 for the 3 tokens it can see (#1, #2 and #3), and −∞ for the 5 hidden ones. So the e^score are 1, 1 and 1 (and 0 for the hidden ones), and each visible token gets 1 ÷ 3 = **⅓**.

</details>

In [ ]:
my_answer = None  # a number, e.g. 0.25
check("4.3", my_answer)

<!-- notebook-only -->

### 4.4 (code) Write `my_softmax(scores)`: e to the power of each score, divided by the total.

<details>
<summary>Show the answer</summary>

```python
def my_softmax(scores):
    exps = np.exp(scores)
    return exps / exps.sum()
```

</details>

<!-- /notebook-only -->

In [ ]:
def my_softmax(scores):
    # your code here
    return None


check("4.4", my_softmax)

### 4.5 Think: why can't `_hungry` (#3) look at `_milk` (#7)?

<details>
<summary>Show the answer</summary>
<!-- choices: *`_milk` comes later: when the toddler writes, the future doesn't exist yet, so it is hidden | `_milk`'s key is 0 | `_hungry` has no query | `_milk` is too far away -->

Because of the **causal mask**: row 3, column 7 of the scores is −∞, so its weight is 0. When the toddler writes, it guesses one token at a time: the tokens after the current one don't exist yet. Training must work the same way, so every token only sees itself and the tokens before it.

</details>

### 4.6 Think: why are the scores divided by √4?

<details>
<summary>Show the answer</summary>
<!-- choices: *To keep the scores from growing too big with long vectors, so that softmax doesn't get too sharp | To make the scores positive | To make the computation faster | Because a query has 2 parts -->

A dot product adds up one product per number, so it tends to grow with the size of the vectors. Big scores make softmax almost all-or-nothing, and learning gets hard. Dividing by √(key size) keeps the scores in a good range, whatever the size.

</details>

## Summary

- **Attention** lets each token look back at the earlier tokens and take what it needs.
- The sentence is a matrix **X** (8 × 10), one row per token. Each token makes a **query** (what am I looking for?), a **key** (what do I have?) and a **value** (what do I give?), all at once with three matrix multiplications: **Q = X · Wq**, **K = X · Wk**, **V = X · Wv** (8 × 4 each).
- To multiply two matrices: each number of the result is the dot product of a row of the left matrix with a column of the right one.
- **Q · Kᵀ ÷ √(key size)** gives the scores (8 × 8): every question meets every label. The future gets −∞ (the **causal mask**). **Softmax** turns each row into shares that add up to 100%: the **weights**.
- **weights · V** mixes the values, **Wo** puts the mix back into the 10 columns, and the result is **added** to X. The output has the same size as X.
- With our hint matrices, `_want` gives 98.8% of its attention to `_hungry`, and gets +0.99 food.
- One set of Wq, Wk, Wv and Wo is a **head**. Real models run many heads at once.
- Next: **Step 5, the transformer block**.